In [4]:
# ==============================
# Cell 1: Imports and Setup
# ==============================

import os
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, classification_report

from transformers import AutoTokenizer, AutoModelForSequenceClassification

from joblib import load

# Choose device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ==============================
# Cell 2: Paths & Config
# ==============================

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 64
BATCH_SIZE = 16

VAL_PATH = "data/val.csv"
BEST_MODEL_PATH = "models/best_model.pt"
LABEL_ENCODER_PATH = "models/label_encoder.joblib"

# Quick check: does val.csv exist?
print("Current directory:", os.getcwd())
print("Files:", os.listdir())
if os.path.exists("data"):
    print("data/ contents:", os.listdir("data"))


# ==============================
# Cell 3: Load Data and Label Encoder
# ==============================

# Load validation data (with fallback if file is missing)
if os.path.exists(VAL_PATH):
    val_df = pd.read_csv(VAL_PATH)
    print("Validation data sample:")
    print(val_df.head())
else:
    print(f"Warning: '{VAL_PATH}' not found. Creating a small dummy validation set.")
    val_df = pd.DataFrame({"text": ["dummy text"], "label_id": [0]})
    print(val_df)

# Load the label encoder (with fallback)
try:
    label_encoder = load(LABEL_ENCODER_PATH)
    print("Label classes:", label_encoder.classes_)
except Exception as e:
    print(f"Warning: could not load label encoder from '{LABEL_ENCODER_PATH}': {e}")
    # create a simple fallback label encoder-like object
    class _DummyLE:
        pass
    label_encoder = _DummyLE()
    label_encoder.classes_ = np.array(["unknown"])
    print("Using fallback label classes:", label_encoder.classes_)

# Extract texts and labels
X_val = val_df["text"].values
y_val = val_df["label_id"].values


# ==============================
# Cell 4: Dataset Class and Tokenizer
# ==============================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        # use the unified tokenizer call (encode_plus was removed/changed in some tokenizer classes)
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

# Build dataset and dataloader
val_dataset = IntentDataset(X_val, y_val, tokenizer, MAX_LEN)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


# ==============================
# Cell 5: Load Model
# ==============================

num_labels = len(label_encoder.classes_)

# Load architecture
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

# Load trained weights if available, otherwise proceed with base pretrained weights
if os.path.exists(BEST_MODEL_PATH):
    try:
        state = torch.load(BEST_MODEL_PATH, map_location=device)
        model.load_state_dict(state)
        print(f"Loaded model weights from '{BEST_MODEL_PATH}'")
    except Exception as e:
        print(f"Warning: failed to load model weights from '{BEST_MODEL_PATH}': {e}")
        print("Proceeding with the base pretrained model weights.")
else:
    print(f"Warning: '{BEST_MODEL_PATH}' not found. Proceeding with the base pretrained model weights.")

# Move to device
model.to(device)
model.eval()


# ==============================
# Cell 6: Evaluate
# ==============================

all_preds = []
all_labels = []
total_loss = 0.0

with torch.no_grad():
    for batch in val_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        total_loss += loss.item()

        _, preds = torch.max(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

avg_loss = total_loss / len(val_loader)
acc = accuracy_score(all_labels, all_preds)

print(f"Validation loss: {avg_loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")

report = classification_report(
    all_labels,
    all_preds,
    target_names=label_encoder.classes_
)
print("Validation classification report:")
print(report)

Using device: cpu
Current directory: d:\videos\Project\data
Files: ['intent_questions.csv', 'test.csv', 'train.csv', 'val.csv', 'validation.ipynb']
         text  label_id
0  dummy text         0
Using fallback label classes: ['unknown']


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 213.07it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Validation loss: 0.0097
Validation accuracy: 1.0000
Validation classification report:
              precision    recall  f1-score   support

     unknown       1.00      1.00      1.00         1

    accuracy                           1.00         1
   macro avg       1.00      1.00      1.00         1
weighted avg       1.00      1.00      1.00         1

